# LangGraph with AgentCore Memory - Human in the Loop (Short term memory)

# LangGraph 与 AgentCore Memory - 人机协作（短期记忆）

## Introduction

## 简介

This notebook demonstrates how to integrate Amazon Bedrock AgentCore Memory capabilities with LangGraph to create **human-in-the-loop** workflows. We'll focus on **short-term memory** persistence combined with the ability to interrupt agent execution for human intervention, creating sophisticated customer support scenarios with seamless handoffs.

本笔记本演示如何将 Amazon Bedrock AgentCore Memory 功能与 LangGraph 集成，以创建**人机协作**工作流。我们将重点关注**短期记忆**持久化与中断代理执行以进行人工干预的能力相结合，创建具有无缝交接的复杂客户支持场景。

## Tutorial Details

## 教程详情

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent usecase       | Customer Support with Human Escalation                                          |
| Agentic Framework   | Langgraph                                                                        |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | AgentCore Short-term Memory, Langgraph Checkpointer, Human-in-the-Loop        |
| Example complexity  | Beginner                                                                     |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 短期对话                                                                          |
| 代理用例             | 带有人工升级的客户支持                                                              |
| 代理框架             | Langgraph                                                                        |
| LLM 模型            | Anthropic Claude Haiku 4.5                                                      |
| 教程组件             | AgentCore 短期记忆、Langgraph 检查点器、人机协作                                    |
| 示例复杂度           | 初级                                                                              |

You'll learn to:
- Create a memory checkpointer with AgentCore Memory for workflow persistence
- Use LangGraph's interrupt mechanism for human-in-the-loop workflows
- Implement tools that can pause execution for human intervention
- Resume agent workflows after human input using LangGraph Commands
- Manage complex customer support scenarios with seamless handoffs

您将学习：
- 使用 AgentCore Memory 创建用于工作流持久化的内存检查点器
- 使用 LangGraph 的中断机制实现人机协作工作流
- 实现可以暂停执行以进行人工干预的工具
- 使用 LangGraph Commands 在人工输入后恢复代理工作流
- 管理具有无缝交接的复杂客户支持场景

### Scenario Context

### 场景背景

In this example, we'll create a "**Customer Support Agent**" that can escalate complex issues to human supervisors. When the agent encounters situations requiring human expertise, it will pause execution, save the current state to AgentCore Memory, and wait for human intervention. The human supervisor can then provide guidance, and the agent will resume with the enhanced context.

在本示例中，我们将创建一个"**客户支持代理**"，它可以将复杂问题升级给人工主管。当代理遇到需要人工专业知识的情况时，它将暂停执行，将当前状态保存到 AgentCore Memory，并等待人工干预。然后人工主管可以提供指导，代理将在增强的上下文中恢复运行。

## Architecture

## 架构

<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## Prerequisites

## 前提条件

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

- Python 3.10+
- 具有适当权限的 AWS 账户
- 具有 AgentCore Memory 适当权限的 AWS IAM 角色
- 访问 Amazon Bedrock 模型

### How the Integration Works

### 集成工作原理

The integration between LangGraph and AgentCore Memory for human-in-the-loop workflows involves:

1. Using AgentCore Memory as a checkpointer backend for persistent state management
2. Implementing interrupt mechanisms that pause execution at specific points
3. Enabling human supervisors to resume workflows with additional context
4. Maintaining conversation history and state across interruptions

用于人机协作工作流的 LangGraph 与 AgentCore Memory 之间的集成包括：

1. 使用 AgentCore Memory 作为持久状态管理的检查点后端
2. 实现在特定点暂停执行的中断机制
3. 使人工主管能够使用额外上下文恢复工作流
4. 在中断期间维护对话历史和状态

This approach creates support workflows where AI agents and human supervisors work together seamlessly.

这种方法创建了 AI 代理和人工主管可以无缝协作的支持工作流。

Let's get started by setting up our environment!

让我们开始设置我们的环境！

In [ ]:
# Install necessary libraries
!pip install -qr requirements.txt

In [ ]:
# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# Imports that enable human-in-the-loop implementation
from langgraph.types import Command, interrupt

In [ ]:
import os
import logging

from bedrock_agentcore.memory import MemoryClient
# Import the AgentCoreMemorySaver that we will use as a checkpointer
from langgraph_checkpoint_aws import AgentCoreMemorySaver

logging.getLogger("support-agent").setLevel(logging.INFO)
region = os.getenv('AWS_REGION', 'us-west-2')

logger = logging.getLogger("support-agent")

## Step 1: Memory Creation

## 步骤 1：创建记忆

In this section, we'll create a memory store using the AgentCore Memory SDK. This memory will serve as the backend for our LangGraph checkpointer and enable persistent human-in-the-loop workflows.

在本节中，我们将使用 AgentCore Memory SDK 创建一个内存存储。这个内存将作为我们 LangGraph 检查点器的后端，并支持持久的人机协作工作流。

In [ ]:
memory_name = "SupportAgent"

client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory["id"]

### AgentCore Memory Configuration

### AgentCore Memory 配置

Now let's configure our AgentCore Memory checkpointer and initialize the LLM:

现在让我们配置 AgentCore Memory 检查点器并初始化 LLM：

- `memory_id` corresponds to our AgentCore Memory resource where checkpoints will be stored
- `region` specifies the AWS region for our resources
- `MODEL_ID` defines the Bedrock model that will power our LangGraph agent

- `memory_id` 对应于我们存储检查点的 AgentCore Memory 资源
- `region` 指定我们资源所在的 AWS 区域
- `MODEL_ID` 定义为我们的 LangGraph 代理提供支持的 Bedrock 模型

We will use the `memory_id` and any additional boto3 client keyword args (in our case, `region`) to instantiate our checkpointer.

我们将使用 `memory_id` 和任何额外的 boto3 客户端关键字参数（在本例中为 `region`）来实例化我们的检查点器。

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# Initialize checkpointer for state persistence
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

## Step 2: Human-in-the-Loop Tool

## 步骤 2：人机协作工具

Let's define the tools our support agent will use. Using the LangGraph `interrupt` type, we can interrupt the agent graph execution to give the chance for a human to intervene and respond to the query to continue execution.

让我们定义我们的支持代理将使用的工具。使用 LangGraph 的 `interrupt` 类型，我们可以中断代理图的执行，为人工干预并响应查询以继续执行提供机会。

In [ ]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b

@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply, human_assistance]

## Step 3: LangGraph Agent Implementation

## 步骤 3：LangGraph 代理实现

Now let's create our support agent using LangGraph's `create_react_agent` builder with our AgentCore Memory checkpointer and human-in-the-loop capabilities:

现在让我们使用 LangGraph 的 `create_react_agent` 构建器和我们的 AgentCore Memory 检查点器以及人机协作功能来创建我们的支持代理：

In [ ]:
# Initialize LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## Step 4: Run the Support Agent

## 步骤 4：运行支持代理

We can now run the agent with our AgentCore Memory checkpointer and human-in-the-loop integration. For this example we will ask explicitly for user assistance. In reality, this could be triggered by several conditions, for example a safety flag may route a conversation to a human if certain keywords are used.

现在我们可以使用 AgentCore Memory 检查点器和人机协作集成来运行代理。在本示例中，我们将明确请求用户协助。在实际情况中，这可以由多种条件触发，例如，如果使用了某些关键词，安全标志可能会将对话路由给人工处理。

### Configuration Setup

### 配置设置

In LangGraph, config is a `RuntimeConfig` that contains attributes that are necessary at invocation time, for example user IDs or session IDs. You can [read additional information here](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/).

在 LangGraph 中，config 是一个 `RuntimeConfig`，包含调用时所需的属性，例如用户 ID 或会话 ID。您可以[在此处阅读更多信息](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)。

For the AgentCore Memory checkpointer (`AgentCoreMemorySaver`), we need to specify:
- `thread_id`: Maps to AgentCore session_id (unique conversation thread)
- `actor_id`: Maps to AgentCore actor_id (user, agent or other identifier)

对于 AgentCore Memory 检查点器（`AgentCoreMemorySaver`），我们需要指定：
- `thread_id`：映射到 AgentCore session_id（唯一的对话线程）
- `actor_id`：映射到 AgentCore actor_id（用户、代理或其他标识符）

### Graph Invoke Input

### 图调用输入

We only need to pass the newest user message in as an argument `inputs`. This could include other state variables as well but for the simple `create_react_agent`, only messages are required.

我们只需要将最新的用户消息作为参数 `inputs` 传入。这也可以包括其他状态变量，但对于简单的 `create_react_agent`，只需要消息即可。

In [ ]:
user_input = "I would like to work with a customer service human agent."
config = {"configurable": {"thread_id": "1", "actor_id": "demo-notebook"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

### Workflow Interruption

### 工作流中断

Notice how execution paused when the human assistance tool was called. Let's inspect the current state to see where the workflow stopped:

请注意，当调用人工协助工具时，执行是如何暂停的。让我们检查当前状态以查看工作流在哪里停止：

In [ ]:
snapshot = graph.get_state(config)
snapshot.next

### Human Supervisor intervention

### 人工主管干预

Now let's act as the human supervisor and provide assistance to resume the workflow using the LangGraph `Command` to send our response. The AgentCore Memory checkpointer has preserved the entire conversation state, that will allow us to resume the chat.

现在让我们扮演人工主管的角色，使用 LangGraph `Command` 发送我们的响应来提供协助以恢复工作流。AgentCore Memory 检查点器已保留了整个对话状态，这将允许我们恢复聊天。

In [ ]:
human_response = (
    "I'm sorry to hear that you are frustrated. Looking at the past conversation history, I can see that you've requested a refund. I've gone ahead and credited it to your account."
)

human_command = Command(resume={"messages": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

## Summary

## 总结

In this notebook, we've demonstrated:

在本笔记本中，我们演示了：

1. How to create an AgentCore Memory resource for human-in-the-loop workflows
2. Building a LangGraph agent with interrupt capabilities
3. Implementing tools that can pause execution for human intervention
4. Using the AgentCoreMemorySaver to persist workflow state during interruptions
5. Resuming agent execution with human-provided context

1. 如何为人机协作工作流创建 AgentCore Memory 资源
2. 构建具有中断功能的 LangGraph 代理
3. 实现可以暂停执行以进行人工干预的工具
4. 使用 AgentCoreMemorySaver 在中断期间持久化工作流状态
5. 使用人工提供的上下文恢复代理执行

This integration showcases the power of combining LangGraph's human-in-the-loop capabilities with AgentCore Memory's robust state persistence to create sophisticated customer support workflows where AI agents and human supervisors work together seamlessly.

这种集成展示了将 LangGraph 的人机协作功能与 AgentCore Memory 强大的状态持久化相结合的能力，以创建 AI 代理和人工主管可以无缝协作的复杂客户支持工作流。

The approach we've demonstrated can be extended to more complex scenarios, including multi-level escalations, specialized human expertise routing, and complex approval workflows.

我们演示的方法可以扩展到更复杂的场景，包括多级升级、专业人工专家路由和复杂的审批工作流。

## Clean up

## 清理

Let's delete the memory to clean up the resources used in this notebook.

让我们删除内存以清理本笔记本中使用的资源。

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)